# Writer's Assistant's with LangGraph

1. **The State:** A unified object holding the `topic`, `current_draft`, `feedback`, and an `is_approved` boolean.
2. **The Drafter Node:** Writes a short paragraph based on the topic and any prior feedback.
3. **The Evaluator Node:** Reads the draft. If it doesn't sound right (e.g. missing British English or sounding too robotic), it sets `is_approved=False` and provides feedback.
4. **The Conditional Edge:** If `is_approved` is False, the graph loops back to the Drafter. If True, it ends!

In [ ]:
from typing import TypedDict

from IPython.display import Image, display
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

load_dotenv(override=True)

### State and Output Schemas

In [ ]:
class GraphState(TypedDict):
    topic: str
    current_draft: str
    feedback: str
    is_approved: bool
    revision_count: int

class Evaluation(BaseModel):
    is_approved: bool = Field(description="True if the draft is perfect and strictly uses British English. False if it needs revision.")
    feedback: str = Field(description="If not approved, specific feedback on what to change.")

### Nodes

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

def drafter_node(state: GraphState):
    topic = state["topic"]
    feedback = state.get("feedback", "")
    
    prompt = f"Write a short, engaging 2-sentence paragraph about '{topic}'. Use British English spelling."
    if feedback:
        prompt += f"\n\nYour previous draft was rejected. Here is the editor's feedback: {feedback}\nPlease revise the draft to fix these issues."
        
    response = llm.invoke(prompt)
    
    return {
        "current_draft": response.content, 
        "revision_count": state.get("revision_count", 0) + 1
    }

def evaluator_node(state: GraphState):
    draft = state["current_draft"]
    
    prompt = f"You are a strict British editor. Review this draft:\n\n'{draft}'\n\nIf you see American words like 'flavor' instead of 'flavour', or 'color' instead of 'colour', reject it. If it sounds like generic AI filler, reject it. Otherwise approve it."
    
    evaluator_llm = llm.with_structured_output(Evaluation)
    evaluation = evaluator_llm.invoke(prompt)
    
    return {
        "is_approved": evaluation.is_approved,
        "feedback": evaluation.feedback
    }

### Graph with Conditional Edges

In [ ]:
def route_draft(state: GraphState):
    if state["is_approved"]:
        return "approved"  # Routes to END
    elif state["revision_count"] > 3:
        print("\n[Max revisions reached, forcing approval]")
        return "approved"
    else:
        return "rejected"  # Routes back to Drafter


workflow = StateGraph(GraphState)

workflow.add_node("Drafter", drafter_node)
workflow.add_node("Evaluator", evaluator_node)

workflow.add_edge(START, "Drafter")

workflow.add_edge("Drafter", "Evaluator")

workflow.add_conditional_edges(
    "Evaluator",
    route_draft,
    {
        "approved": END,
        "rejected": "Drafter"
    }
)

graph = workflow.compile()

# visualisation of the flow
display(Image(graph.get_graph().draw_mermaid_png()))

### Executing the Graph

In [ ]:
topic = "The unique flavor and color of Egusi soup."  # using American spelling on purpose

initial_state = {
    "topic": topic, 
    "current_draft": "", 
    "feedback": "", 
    "is_approved": False, 
    "revision_count": 0
}

print(f"\n--- Starting the LangGraph Loop for topic: '{topic}' ---\n")

for output in graph.stream(initial_state):
    for node_name, state_update in output.items():
        if node_name == "Drafter":
            print(f"[DRAFTER (Rev {state_update['revision_count']})] Draft:\n\"{state_update['current_draft']}\"\n")
        elif node_name == "Evaluator":
            status = 'APPROVED' if state_update['is_approved'] else 'REJECTED'
            print(f"[EVALUATOR] Decision: {status}\nFeedback: \"{state_update['feedback']}\"\n")
            print("-" * 40 + "\n")